<a href="https://colab.research.google.com/github/ayoubtabti0000-creator/programmation-python/blob/main/Copie_de_Correction_Tp1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1**. Chargement des données


In [ ]:
# À compléter : Importer les librairies nécessaires
import pandas as pd
import plotly.express as px

# Charger le dataset (disponible sur Kaggle)
df = pd.read_csv("SeoulBikeData.csv", encoding='latin1')

# Afficher les 5 premières lignes

In [ ]:
df

,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,30/11/2018,1003,19,4.2,34,2.6,1894,-10.3,0.0,0.0,0.0,Autumn,No Holiday,Yes
8756,30/11/2018,764,20,3.4,37,2.3,2000,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes
8757,30/11/2018,694,21,2.6,39,0.3,1968,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes
8758,30/11/2018,712,22,2.1,41,1.0,1859,-9.8,0.0,0.0,0.0,Autumn,No Holiday,Yes


# **2**. EDA (Analyse Exploratoire)


Tâche 1 : Afficher un histogramme de la variable cible Rented Bike Count avec Plotly.



In [ ]:
# À compléter
fig = px.histogram(df, x="Rented Bike Count", title="Distribution de la variable cible")
fig.show()

Tâche 2 : Visualiser la demande moyenne par heure avec un graphique linéaire.



In [ ]:
hourly_demand = df.groupby(by='Hour')["Rented Bike Count"].mean()


In [ ]:
hourly_demand.index

Index([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23],
      dtype='int64', name='Hour')

In [ ]:
# À compléter : Grouper par heure et calculer la moyenne
fig = px.line(hourly_demand, x=hourly_demand.index, y="Rented Bike Count")
fig.show()

# **3. Préprocessing**


Tâche 3 : Convertir la colonne Date en datetime et extraire le mois.



In [ ]:
df['Date'] = pd.to_datetime(df['Date'],format="%d/%m/%Y")


In [ ]:
df['Month'] = df['Date'].dt.month

Tâche 4 : Séparer les features (X) et la target (y), puis split train/test.



In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(['Rented Bike Count', 'Date'], axis=1)
y = df['Rented Bike Count']

# À compléter : Split avec 20% de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)



# **4**. Feature Engineering



Tâche 5 : Créer un pipeline de preprocessing avec StandardScaler pour les numériques et OneHotEncoder pour les catégorielles.


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

num_features = ['Temperature(°C)', 'Hour', 'Humidity(%)']
cat_features = ['Seasons', 'Holiday']
#cat2_features=["moyenne"] #exemple

preprocessor = ColumnTransformer([
    ('Standardization des valeurs numériques', StandardScaler(), num_features),
    ('codage en onehot encoding', OneHotEncoder(), cat_features),
    #('codage des variables comparatives' ,LabelEncoder(),cat2_features )
    #...
])

X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# **5**. Construction du Modèle


Tâche 6 : Créer un modèle Sequential avec 2 couches Dense (128 et 64 neurones) et compiler avec Adam.



In [ ]:
X_train_preprocessed.shape[1]

9

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu', input_shape=(X_train_preprocessed.shape[1],)),
    tf.keras.layers.Dense(128, activation='relu'),

        tf.keras.layers.Dense(64, activation='relu'),

        tf.keras.layers.Dense(32, activation='relu'),

        tf.keras.layers.Dense(16, activation='relu'),

    tf.keras.layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 256)            │         2,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,337 (181.00 KB)

 Trainable params: 46,337 (181.00 KB)

 Non-trainable params: 0 (0.00 B)

6. Entraînement et Évaluation


Tâche 7 : Entraîner le modèle avec 50 epochs et un batch_size=32.



In [ ]:
history = model.fit(
    X_train_preprocessed, y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.2 #(X_val,y_val)
)

Epoch 1/150
176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 417766.8750 - mae: 441.4629 - val_loss: 205902.8750 - val_mae: 316.0730
Epoch 2/150
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 204658.2812 - mae: 311.5542 - val_loss: 202852.0469 - val_mae: 313.9063
Epoch 3/150
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 201902.0938 - mae: 308.6763 - val_loss: 207320.4219 - val_mae: 322.2807
Epoch 4/150
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 197546.7188 - mae: 304.4183 - val_loss: 198884.0156 - val_mae: 311.1667
Epoch 5/150
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 190692.9844 - mae: 296.6552 - val_loss: 186790.7969 - val_mae: 296.9805
Epoch 6/150
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 182124.9844 - mae: 289.6659 - val_loss: 179614.6562 - val_mae: 288.9734
Epoch 7/150
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 173433.7656 - mae: 280.8787 - val_loss: 169551.6406 - val_mae: 285.5169
Epoch 8/150
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 163711.8

Tâche 8 : Calculer le RMSE et R² sur le test set.



In [ ]:
from sklearn.metrics import root_mean_squared_error, r2_score

y_pred = model.predict(X_test_preprocessed).flatten()
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"RMSE: {rmse:.2f}, R²: {r2:.2f}")

55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
RMSE: 325.52, R²: 0.75
